In [ ]:
import cv2
import mediapipe as mp
import numpy as np

# --- 📌 新增：計算角度的函式 ---
def calculate_angle(a, b, c):
    """
    計算 a, b, c 三點的夾角 (b 是頂點)
    輸入格式為 [x, y] 的串列
    """
    a = np.array(a) # 點 1 (例如肩膀)
    b = np.array(b) # 點 2 (例如手肘 - 頂點)
    c = np.array(c) # 點 3 (例如手腕)
    
    # 計算弧度並轉成角度
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0/np.pi)
    
    if angle > 180.0:
        angle = 360 - angle
        
    return angle
# -----------------------------

print("=== 選單 ===")
print("1: 大樹式")
print("2: 深蹲")

choice = input("請輸入你的選項 (1/2): ")

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(image)
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        if results.pose_landmarks:
            # 畫出骨架
            mp_drawing.draw_landmarks(
                image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
            
            # --- 📌 新增：抓取座標與邏輯判斷 ---
            try:
                landmarks = results.pose_landmarks.landmark
                
                # 1. 抓取需要的四個點
                hip = [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x, 
                    landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y]
                shoulder = [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, 
                            landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y]
                elbow = [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, 
                        landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y]
                wrist = [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x, 
                        landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y]
                knee = [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, 
                        landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y]
                ankle = [landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x, 
                        landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y]
               

                # 2. 計算兩個角度
                elbow_angle = calculate_angle(shoulder, elbow, wrist) # 手肘伸直程度
                shoulder_angle = calculate_angle(hip, shoulder, elbow) # 手臂抬高程度
                knee_angle = calculate_angle(hip, knee, ankle) # 腳抬高程度
                leg_angle = calculate_angle(shoulder, hip, knee) # 腳抬高程度
                
                match choice:
                    # 3. 判斷姿勢是否正確 (假設目標是手伸直，角度需大於 160 度)
                    # 3. 雙重判斷 (設定一個容錯範圍，例如 90度 ± 15度)

                    case '1': #大樹:手平舉，腳舉起貼在膝蓋上
                        if elbow_angle < 160:
                            # print("錯誤：手肘彎曲了！請伸直。")
                            # 參數：圖片, 文字, 座標, 字體, 大小, BGR顏色(紅色), 粗細, 線條種類
                            cv2.putText(image, 'ERROR: Straighten your arm!', (50, 50), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)
                        elif shoulder_angle < 75:
                            # print("錯誤：手臂掉下來了！請抬高到水平。")
                            # 參數：圖片, 文字, 座標, 字體, 大小, BGR顏色(紅色), 粗細, 線條種類
                            cv2.putText(image, 'ERROR: Rise', (50, 50), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)
                        elif shoulder_angle > 105:
                            # print("錯誤：手臂舉太高了！請放平。")
                            # 參數：圖片, 文字, 座標, 字體, 大小, BGR顏色(紅色), 粗細, 線條種類
                            cv2.putText(image, 'ERROR: Put Down', (50, 50), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)
                        else:
                            # print("完美平舉！")
                            # 參數：圖片, 文字, 座標, 字體, 大小, BGR顏色(紅色), 粗細, 線條種類
                            cv2.putText(image, 'PERFECT', (50, 50), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA)


                        if leg_angle > 115:
                            # print("錯誤：再抬高")
                            # 參數：圖片, 文字, 座標, 字體, 大小, BGR顏色(紅色), 粗細, 線條種類
                            cv2.putText(image, 'ERROR: Raise Your Leg!', (50, 100), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)
                        elif leg_angle < 75:
                            # print("錯誤：腳低一點")
                            # 參數：圖片, 文字, 座標, 字體, 大小, BGR顏色(紅色), 粗細, 線條種類
                            cv2.putText(image, 'ERROR: Down Your Leg', (50, 100), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)
                        elif knee_angle > 75:
                            # print("錯誤：腳彎")
                            # 參數：圖片, 文字, 座標, 字體, 大小, BGR顏色(紅色), 粗細, 線條種類
                            cv2.putText(image, 'ERROR: Bend Your leg', (50, 100), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)
                        else:
                            # print("完美平舉！")
                            # 參數：圖片, 文字, 座標, 字體, 大小, BGR顏色(紅色), 粗細, 線條種類
                            cv2.putText(image, 'LEG PERFECT', (50, 100), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA)


                    case '2': #深蹲(不管手)
                        if knee_angle < 120:
                            # print("錯誤：蹲低一點")
                            cv2.putText(image, 'ERROR: Squat low', (50, 100), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)
                        else:
                            # print("完美平舉！")
                            cv2.putText(image, 'PERFECT', (50, 100), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA)

            except Exception as e: #try跟except要對齊
                pass
                    # -----------------------------

        cv2.imshow('Yoga Test', image)
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

=== 選單 ===
1: 大樹式
2: 深蹲
